# A1 Pairs/OU — 02 · Cointegration Screen

Find pairs with a **stationary spread**, the precondition for OU mean-reversion.
Method: Engle-Granger — OLS hedge ratio β on log-prices, ADF test on the residual.
We split **in-sample / out-of-sample** and require cointegration to hold in *both*.

> **Multiple-testing discipline:** we test every pair, so the best one looks good by
> luck. We record the number of pairs tested (`N_TRIALS`) and deflate for it in NB3
> (DSR). Do **not** cherry-pick the prettiest spread here and forget the others existed.

In [ ]:
import os, sys, itertools
import numpy as np, pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("../.."))          # repo root for basecamp_recon

OUT = "out"
panel = pd.read_parquet(f"{OUT}/panel.parquet")
stats = pd.read_csv(f"{OUT}/symbol_stats.csv", index_col=0)
logp  = np.log(panel)
print("panel:", panel.shape, "| span:", panel.index.min(), "->", panel.index.max())
stats

In [ ]:
# ── in-sample / out-of-sample split (chronological) ─────────────────────────────
SPLIT = 0.6
n = len(logp); k = int(n*SPLIT)
IS, OOS = logp.iloc[:k], logp.iloc[k:]
print(f"IS bars={len(IS)}  OOS bars={len(OOS)}")

In [ ]:
def eg_test(a, b):
    """Engle-Granger on a,b (log-price series). Returns hedge beta + ADF p on residual."""
    x = sm.add_constant(b.values)
    beta = sm.OLS(a.values, x).fit().params         # [const, slope]
    resid = a.values - (beta[0] + beta[1]*b.values)
    adf_p = adfuller(resid, maxlag=1, autolag=None)[1]
    return beta[1], adf_p, resid

def rolling_beta_std(a, b, win=120):
    cov = a.rolling(win).cov(b); var = b.rolling(win).var()
    return float((cov/var).std())

rows = []
pairs = list(itertools.combinations(panel.columns, 2))
for x, y in pairs:
    s = panel[[x, y]].dropna()
    if len(s) < 500:                                  # need enough overlap
        continue
    ai, bi = np.log(s[x]).loc[IS.index.intersection(s.index)], np.log(s[y]).loc[IS.index.intersection(s.index)]
    ao, bo = np.log(s[x]).loc[OOS.index.intersection(s.index)], np.log(s[y]).loc[OOS.index.intersection(s.index)]
    if len(ai) < 300 or len(ao) < 200:
        continue
    beta_is, p_is, _   = eg_test(ai, bi)
    beta_oos, p_oos, _ = eg_test(ao, bo)
    rows.append({"x": x, "y": y, "beta_is": round(beta_is,3), "beta_oos": round(beta_oos,3),
                 "adf_p_is": round(p_is,4), "adf_p_oos": round(p_oos,4),
                 "beta_drift": round(rolling_beta_std(np.log(s[x]), np.log(s[y])),3),
                 "corr": round(np.log(s[x]).diff().corr(np.log(s[y]).diff()),3),
                 "n": len(s)})
N_TRIALS = len(rows)
screen = pd.DataFrame(rows).sort_values("adf_p_oos")
print("pairs tested (N_TRIALS):", N_TRIALS)
screen

In [ ]:
# ── candidates: cointegrated IS *and* OOS, stable beta ──────────────────────────
ALPHA = 0.05
cand = screen[(screen.adf_p_is < ALPHA) & (screen.adf_p_oos < ALPHA)].copy()
cand = cand.sort_values(["adf_p_oos", "beta_drift"])
print(f"{len(cand)} pair(s) cointegrated in BOTH halves at p<{ALPHA}:")
cand

In [ ]:
# ── visualise the top candidate spreads ─────────────────────────────────────────
top = cand.head(4) if len(cand) else screen.head(4)
fig, axes = plt.subplots(len(top), 1, figsize=(13, 2.6*len(top)), squeeze=False)
for ax, (_, r) in zip(axes[:,0], top.iterrows()):
    s = panel[[r.x, r.y]].dropna()
    beta = sm.OLS(np.log(s[r.x]).values, sm.add_constant(np.log(s[r.y]).values)).fit().params
    spread = np.log(s[r.x]) - (beta[0] + beta[1]*np.log(s[r.y]))
    ax.plot(spread.values); ax.axhline(spread.mean(), color="k", lw=.6)
    ax.set_title(f"{r.x}-{r.y}  adf_p_oos={r.adf_p_oos}  beta_drift={r.beta_drift}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── save candidates + trial count for NB3 ───────────────────────────────────────
cand.to_csv(f"{OUT}/candidate_pairs.csv", index=False)
pd.Series({"N_TRIALS": N_TRIALS}).to_csv(f"{OUT}/screen_meta.csv")
print("wrote candidate_pairs.csv (N_TRIALS=%d)" % N_TRIALS)

**Read it honestly:** a pair passing here is a *candidate*, not an edge. Beta drift
matters as much as the ADF p-value — a spread that's stationary but whose hedge ratio
wanders is not tradeable. If **zero** pairs pass in both halves, that's a clean *kill*
of A1 on this data; don't lower ALPHA to manufacture survivors. Next: `03_ou_fit_and_backtest.ipynb`.